# CS336 Assignment 1 on Colab

依次运行各单元格。先在「运行时 → 更改运行时类型」启用 GPU。
默认做 100 步 smoke run。正式训练需把下面的 MODE 改为 full。
代码来自自己的 GitHub，训练文件放在 Colab 本地磁盘，checkpoint 和日志放在 Google Drive。


In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/mwang693-rgb/cs336-assignment1.git"
ROOT = Path("/content/cs336-assignment1")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "einops", "regex", "matplotlib"], check=True)

import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "请先启用 GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

Code revision: c99bfc1
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
GPU: NVIDIA A100-SXM4-40GB


In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/cs336_assignment1")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 选择实验
smoke 为短跑。full 的默认预算为 327,680,000 token。四层、512 维、16 头、SwiGLU 1344、词表 10K、上下文 256。

In [3]:
MODE = "smoke"  # 改成 "full" 开始正式 baseline
BATCH_SIZE = 64 if "A100" in torch.cuda.get_device_name(0) else 16
MAX_LR = 3e-4
RUN_NAME = "smoke" if MODE == "smoke" else "tinystories_lr3e-4"
RESUME = False  # 恢复同一场训练时改成 True

assert MODE in ("smoke", "full")
assert BATCH_SIZE > 0
CONTEXT = 256
STEPS = 100 if MODE == "smoke" else 327680000 // (BATCH_SIZE * CONTEXT)
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Steps:", STEPS, "Processed tokens:", STEPS * BATCH_SIZE * CONTEXT)
print("Results:", RUN_DIR)

Steps: 100 Processed tokens: 1638400
Results: /content/drive/MyDrive/cs336_assignment1/runs/smoke


## 准备 TinyStories token 文件
首次下载并编码可能较慢。复用仓库里的 10K BPE tokenizer，编码结果也保存到 Drive，以后可以直接复制。

In [ ]:
import shutil

DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
TOKEN_CACHE = DRIVE_ROOT / "tokens"
TOKEN_CACHE.mkdir(exist_ok=True)

for split, source in (
    ("train", "TinyStoriesV2-GPT4-train.txt"),
    ("valid", "TinyStoriesV2-GPT4-valid.txt"),
):
    encoded = DATA / f"tinystories_{split}.bin"
    cached = TOKEN_CACHE / encoded.name
    if cached.exists():
        shutil.copy2(cached, encoded)
        print("Copied cached tokens:", encoded.name)
        continue

    raw = DATA / source
    url = f"https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/{source}"
    subprocess.run(["wget", "-c", "-O", str(raw), url], check=True)
    subprocess.run([
        sys.executable, "-m", "cs336_basics.prepare_data", "encode", str(raw),
        "--vocab", "vocab.pkl", "--merges", "merges.pkl", "--output", str(encoded),
    ], check=True)
    shutil.copy2(encoded, cached)

## 开始训练
运行本单元格才会启动训练。100 步 smoke run 用来检查 loss、显存和文件保存。full 是从头训练的新实验。需要恢复时，保持相同配置并设置 RESUME=True。

In [ ]:
import json

config = {
    "mode": MODE, "batch_size": BATCH_SIZE, "steps": STEPS, "max_lr": MAX_LR,
    "vocab_size": 10000, "context_length": 256, "d_model": 512,
    "d_ff": 1344, "num_heads": 16, "num_layers": 4, "rope_theta": 10000,
}
checkpoint = RUN_DIR / "checkpoint.pt"
config_path = RUN_DIR / "config.json"
if RESUME:
    assert checkpoint.exists(), "没有找到 checkpoint"
    assert json.loads(config_path.read_text()) == config, "恢复时保持原配置"
else:
    assert not checkpoint.exists(), "这个 run 已有 checkpoint，请改 RUN_NAME 或设置 RESUME=True"
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

command = [
    sys.executable, "-u", "-m", "cs336_basics.train",
    "--train-data", str(DATA / "tinystories_train.bin"),
    "--valid-data", str(DATA / "tinystories_valid.bin"),
    "--checkpoint", str(checkpoint),
    "--log", str(RUN_DIR / "metrics.jsonl"),
    "--device", "cuda", "--batch-size", str(BATCH_SIZE), "--steps", str(STEPS),
    "--warmup-steps", str(10 if MODE == "smoke" else min(1000, STEPS // 10)),
    "--max-lr", str(MAX_LR), "--min-lr", str(MAX_LR / 10),
    "--eval-every", str(10 if MODE == "smoke" else 200),
    "--eval-batches", str(4 if MODE == "smoke" else 16),
    "--save-every", str(100 if MODE == "smoke" else 1000),
]
if RESUME:
    command.extend(["--resume", str(checkpoint)])
subprocess.run(command, check=True)

## 查看 loss 曲线和 perplexity
从 Drive 的 metrics.jsonl 读取。PPL = exp(validation loss)。曲线里的最低观测值与最终 checkpoint 的指标可能不同。

In [ ]:
import math
import matplotlib.pyplot as plt

records = [json.loads(line) for line in (RUN_DIR / "metrics.jsonl").read_text().splitlines() if line.strip()]
steps = [r["step"] for r in records]
plt.figure(figsize=(9, 4))
plt.plot(steps, [r["train_loss"] for r in records], label="Training loss")
plt.plot(steps, [r["valid_loss"] for r in records], label="Validation loss")
plt.xlabel("Completed optimizer steps")
plt.ylabel("Cross-entropy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RUN_DIR / "loss.png", dpi=160)
plt.show()
last = records[-1]
print("Last logged validation loss:", last["valid_loss"])
print("Last logged validation PPL:", math.exp(last["valid_loss"]))
print("Best observed validation loss:", min(r["valid_loss"] for r in records))

## 用 checkpoint 生成文本
smoke 模型的输出质量通常很低。full 模型训练完成后，用同一个单元格检查续写。

In [ ]:
from cs336_basics.Transformer_lm import Transformer_lm
from cs336_basics.Tokenizer import Tokenizer
from cs336_basics.Decoderv2 import Decoder

tokenizer = Tokenizer.from_files("vocab.pkl", "merges.pkl", ["<|endoftext|>"])
model = Transformer_lm(512, 16, 1344, 10000, 256, 4, 10000).to("cuda")
saved = torch.load(checkpoint, map_location="cuda", weights_only=True)
model.load_state_dict(saved["model"])
model.eval()

PROMPT = "Once upon a time"
ids = torch.tensor([tokenizer.encode(PROMPT)], dtype=torch.long, device="cuda")
eos_id = tokenizer.special_ids["<|endoftext|>"]
new_ids = Decoder(model, ids, maxtimes=256, eos_id=eos_id, temperature=0.8, p=0.9)
continuation = tokenizer.decode(new_ids[0].tolist())
print(PROMPT + continuation)
(RUN_DIR / "generation.txt").write_text(PROMPT + continuation, encoding="utf-8")